In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
# Load Dataset
data_path = "/kaggle/input/go-emotions-google-emotions-dataset/go_emotions_dataset.csv"
data = pd.read_csv(data_path)

In [3]:
# Drop unnecessary columns
data = data.drop(columns=['id', 'example_very_unclear'])

In [4]:
# Preprocessing
# Combine columns of emotions into a single list of labels per text
data['labels'] = data.iloc[:, 1:].values.tolist()
data = data[['text', 'labels']]

# Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'].tolist(),
    data['labels'].tolist(),
    test_size=0.2,
    random_state=42
)

In [5]:
# Initialize tokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
# Dataset class
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = torch.tensor(self.labels[idx], dtype=torch.float)

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

In [7]:
# Create datasets
max_len = 128
train_dataset = EmotionDataset(train_texts, train_labels, tokenizer, max_len)
test_dataset = EmotionDataset(test_texts, test_labels, tokenizer, max_len)

In [8]:
# DataLoader
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [9]:
# Model
model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", 
                                                            num_labels=28, 
                                                            problem_type="multi_label_classification")

model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768,

In [10]:
# Optimizer and Loss
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss

# Compute class weights
labels_array = np.array(data['labels'].tolist())
label_sums = labels_array.sum(axis=0)
class_weights = torch.tensor(
    [len(labels_array) / (label_sum + 1e-6) for label_sum in label_sums],
    dtype=torch.float
).to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

loss_fn = BCEWithLogitsLoss(pos_weight=class_weights)
optimizer = AdamW(model.parameters(), lr=2e-5)

In [11]:
# Training loop
def train_model(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [12]:
# Evaluation loop
def evaluate_model(model, dataloader, device):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.sigmoid(logits) > 0.5

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    return all_labels, all_preds

In [13]:
# Training and evaluation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 3

for epoch in range(epochs):
    train_loss = train_model(model, train_loader, optimizer, loss_fn, device)
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss}")

Epoch 1/3, Loss: 0.9841369986308386
Epoch 2/3, Loss: 0.8448342515723178
Epoch 3/3, Loss: 0.7908934184410562


In [14]:
# Save model
output_dir = "xlm-roberta_emotion_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('xlm-roberta_emotion_model/tokenizer_config.json',
 'xlm-roberta_emotion_model/special_tokens_map.json',
 'xlm-roberta_emotion_model/sentencepiece.bpe.model',
 'xlm-roberta_emotion_model/added_tokens.json')

In [15]:
!zip -r xlm-roberta.zip /kaggle/working/xlm-roberta_emotion_model

  adding: kaggle/working/xlm-roberta_emotion_model/ (stored 0%)
  adding: kaggle/working/xlm-roberta_emotion_model/special_tokens_map.json (deflated 52%)
  adding: kaggle/working/xlm-roberta_emotion_model/tokenizer_config.json (deflated 76%)
  adding: kaggle/working/xlm-roberta_emotion_model/config.json (deflated 65%)
  adding: kaggle/working/xlm-roberta_emotion_model/model.safetensors (deflated 28%)
  adding: kaggle/working/xlm-roberta_emotion_model/sentencepiece.bpe.model (deflated 49%)


In [16]:
# Define emotion labels explicitly
emotion_labels = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring", 
    "confusion", "curiosity", "desire", "disappointment", "disapproval", 
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief", 
    "joy", "love", "nervousness", "optimism", "pride", "realization", 
    "relief", "remorse", "sadness", "surprise", "neutral"
]

# Evaluate on test set
y_true, y_pred = evaluate_model(model, test_loader, device)

# Print the classification report
print(classification_report(np.array(y_true), np.array(y_pred), target_names=emotion_labels))

                precision    recall  f1-score   support

    admiration       0.28      0.88      0.42      3456
     amusement       0.28      0.87      0.43      1891
         anger       0.15      0.80      0.26      1628
     annoyance       0.16      0.71      0.27      2722
      approval       0.14      0.75      0.23      3418
        caring       0.09      0.85      0.16      1147
     confusion       0.15      0.82      0.25      1463
     curiosity       0.27      0.85      0.40      1941
        desire       0.10      0.80      0.17       758
disappointment       0.09      0.81      0.16      1671
   disapproval       0.16      0.75      0.26      2289
       disgust       0.10      0.78      0.18      1074
 embarrassment       0.05      0.67      0.10       502
    excitement       0.08      0.83      0.14      1121
          fear       0.09      0.86      0.16       625
     gratitude       0.42      0.91      0.57      2330
         grief       0.01      0.84      0.03  

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [17]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
import torch

# Define the emotion labels
emotion_labels = ["admiration", "amusement", "anger", "annoyance", "approval", "caring", 
                  "confusion", "curiosity", "desire", "disappointment", "disapproval", 
                  "disgust", "embarrassment", "excitement", "fear", "gratitude", 
                  "grief", "joy", "love", "nervousness", "optimism", "pride", 
                  "realization", "relief", "remorse", "sadness", "surprise", "neutral"]

# Load the tokenizer and model (done once to improve efficiency)
tokenizer = XLMRobertaTokenizer.from_pretrained("./xlm-roberta_emotion_model")
model = XLMRobertaForSequenceClassification.from_pretrained("./xlm-roberta_emotion_model")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_emotions(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    
    # Move input tensors to the same device as the model
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    # Get predictions
    with torch.no_grad():  # Disable gradient calculations for inference
        outputs = model(**inputs)
    
    predictions = outputs.logits
    
    # Convert predictions to probabilities
    probs = torch.softmax(predictions, dim=1).detach().cpu().numpy()[0]
    
    # Create a dictionary of emotions and their corresponding probabilities
    emotion_probs = {label: prob for label, prob in zip(emotion_labels, probs)}
    
    # Sort the emotions by probability in descending order
    sorted_emotions = sorted(emotion_probs.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_emotions

# Test the function with a new text
text = "je suis triste, j'ai perdu mon ami"
predictions = predict_emotions(text)

# Print the sorted emotions with probabilities
for emotion, prob in predictions:
    print(f"{emotion}: {prob:.4f}")

sadness: 0.6744
grief: 0.1689
remorse: 0.0562
disappointment: 0.0473
caring: 0.0088
nervousness: 0.0067
realization: 0.0051
joy: 0.0029
neutral: 0.0027
embarrassment: 0.0024
relief: 0.0023
fear: 0.0022
love: 0.0021
annoyance: 0.0019
approval: 0.0018
anger: 0.0018
excitement: 0.0017
optimism: 0.0017
disgust: 0.0015
gratitude: 0.0014
disapproval: 0.0013
pride: 0.0011
surprise: 0.0011
desire: 0.0010
amusement: 0.0008
admiration: 0.0006
curiosity: 0.0002
confusion: 0.0002
